EasyUC: Using EasyCrypt to Mechanize Proofs of
Universally Composable Security∗
Ran Canetti† Alley Stoughton‡ Mayank Varia§
May 29, 2019
https://eprint.iacr.org/2019/582.pdf

A EasyCrypt Module for Making Interfaces
This appendix contains the EasyCrypt definition of the module for making an interface out of a
functionality and an adversary.14 (The .` syntax selects the nth component of a tuple.)

In [ ]:
module MI (Func : FUNC, Adv : FUNC) : INTER = {
var func, adv : addr
var in guard : int fset
proc init(func adv : addr, in guard : int fset) : unit = {
func ← func ; adv ← adv ; in guard ← in guard ;
Func.init(func, adv);
Adv.init(adv, [ ]);
}
proc loop(m : msg) : msg option = {
var mod : mode; var pt1, pt2 : port; var u : univ;
var addr1 : addr; var n1 : int;
var r : msg option ← None;
var not done : bool ← true;
(∗ loop invariant in terms of m:
not done ⇒
func ≤ m.`2.`1 ∨
m.`1 = Adv ∧ m.`2.`1 = adv ∗)
while (not done) {
(mod, pt1, pt2, u) ← m; (addr1, n1) ← pt1;
if (func ≤ addr1) {
r <@ Func.invoke(m);
if (r = None) {
not done ← false;
}
else {
m ← oget r; (∗ next iteration, if any, will use m ∗)
(mod, pt1, pt2, u) ← m; (addr1, n1) ← pt1;
if (func ≤ addr1) {
r ← None; not done ← false;
}
elif (mod = Dir) {
not done ← false;
if (adv ≤ addr1) {
r ← None;
}
}
elif (addr1 6= adv ∨ n1 = 0) {
r ← None; not done ← false;
}
}
}
else { (∗ addr1 = adv ∗)
r <@ Adv.invoke(m);
if (r = None) {
not done ← false;
}
else {
m ← oget r; (∗ next iteration, if any, will use m ∗)
(mod, pt1, pt2, u) ← m; (addr1, n1) ← pt1;
if (adv ≤ addr1 ∨ mod = Dir) {
r ← None; not done ← false;
}
elif (! func ≤ addr1) {
not done ← false;
}
}
}
}
return r;
}
proc invoke(m : msg) : msg option = {
var mod : mode; var pt1, pt2 : port; var u : univ;
var addr1, addr2 : addr; var n1 : int;
var r : msg option;
(mod, pt1, pt2, u) ← m; (addr1, n1) ← pt1;
if (func ≤ addr1 ∧ mod = Dir ∨
addr1 = adv ∧ mod = Adv ∧ (n1 = 0 ∨ n1 \in in guard)) {
r <@ loop(m);
}
else {
r ← None;
}
return r;
}
}

B EasyCrypt Composed Environment Module
This appendix contains the EasyCrypt definition of the composed environment module used as
part of the lifting of key-exchange security to SMCReal.
15

In [ ]:
module CompEnv (Env : ENV, Inter : INTER) = {
var stub st : msg option
var func : addr
var adv : addr
var in guard low : int fset
module StubKE : FUNC = {
proc init(func adv : addr) : unit = { }
proc invoke(m : msg) : msg option = {
var mod : mode; var pt1, pt2 : port; var u : univ;
var addr1 : addr; var n1 : int;
var r : msg option;
if (stub st 6= None) {
r ← stub st; stub st ← None;
}
else {
r <@ Inter.invoke(m);
if (r 6= None) {
m ← oget r; (mod, pt1, pt2, u) ← m; (addr1, n1) ← pt1;
if (mod = Adv) {
stub st ← Some m;
(∗ only mode and destination port matter (destination port id
must not be 0) ∗)
r ← Some (Adv, (adv, 1), (func ++ [2], 1), UnivUnit);
}
}
}
return r;
}
}
module StubAdv : FUNC = {
proc init(func adv : addr) : unit = { }
proc invoke(m : msg) : msg option = {
var mod : mode; var pt1, pt2 : port; var u : univ;
var addr1 : addr; var n1 : int;
var r : msg option;
if (stub st 6= None) {
r ← stub st; stub st ← None;
}
else {
r <@ Inter.invoke(m);
if (r 6= None) {
m ← oget r; (mod, pt1, pt2, u) ← m; (addr1, n1) ← pt1;
if (mod = Dir) {
stub st ← Some m;
(∗ only mode and destination address matter ∗)
r ← Some (Adv, (func ++ [2], 1), (adv, 1), UnivUnit);
}
}
}
return r;
}
}
(∗ func will end with 2 ∗)
proc main(func adv : addr, in guard : int fset) : bool = {
var b : bool;
stub st ← None;
func ← take (size func − 1) func ; adv ← adv ;
b <@ Exper(MI'(SMCReal(StubKE), StubAdv), Env).main(func, adv, in guard low);
return b;
}
}.

C Symbolic Evaluation in EasyCrypt
This appendix contains a simple example of how one can carry out symbolic evaluation in EasyCrypt. It involves communication of integers between two entities, A and B, plus an environment.
There is a type dest of destinations, whose elements are the addresses of the two entities plus
the environment:

In [ ]:
type dest = [A | B | Env].

In [ ]:
An entity is a module with a procedure f that transforms an integer into a new integer, along with
the destination to which it should be sent:

In [ ]:
module type ENT = {
proc f(x : int) : dest ∗ int
}.

The routing loop module, Loop, is parameterized by two entities—one for A and one for B:

In [ ]:
module Loop(EntA : ENT, EntB : ENT) = {
proc loop(d : dest, x : int) : int = {
while (d 6= Env) {
if (d = A) {
(d, x) <@ EntA.f(x);
}
else { (∗ d = B ∗)
(d, x) <@ EntB.f(x);
}
}
return x;
}
}.

Its loop procedure takes in an initial destination d and integer x. Its body is a while loop that lets
A and B communicate with each other, until the point where the currently invoked entity decides
to return to the environment.
A and B are implemented as follows:

In [ ]:
module EntA : ENT = {
proc f(x : int) : dest ∗ int = {
x ← x + 1;
return (if 5 ≤ x then Env else B, x);
}
}.
module EntB : ENT = {
proc f(x : int) : dest ∗ int = {
x ← x ∗ 2;
return (if 5 ≤ x then Env else A, x);
}
}.

A increments its input by one, and asks to send the result to B. B doubles its input, and asks to
send the result to A. But both A and B have exceptions: they ask to send to the environment
results that are at least 5.
We can prove the following lemma using symbolic evaluation:

lemma l :
phoare[Loop(EntA, EntB).loop : d = A ∧ x = 1 =⇒ res = 5] = 1%r.
proof.
proc; simplify.
rcondt 1; first auto.
rcondt 1; first auto.
inline (1) EntA.f.
sp.
rcondt 1; first auto.
rcondf 1; first auto.
inline (1) EntB.f.
sp.
rcondt 1; first auto.
rcondt 1; first auto.
inline (1) EntA.f.
sp.
rcondf 1; first auto.
auto.
qed.


The lemma says that if we begin by giving A the input 1, then eventually the result 5 is returned
to the environment (this happens with probability 1).
In what follows, we’ll show the intermediate goals of the proof of this lemma. After applying
the proc (procedure) tactic and simplifying the precondition, we have:


In [ ]:
pre = d = A ∧ x = 1
while (d 6= Env) {
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
}
post = x = 5

Because EasyCrypt’s auto tactic will be able to prove that the while loop’s boolean expression
is true (follows from the precondition), we can use the rcondt (reduce conditional, when true) and
auto tactics

In [ ]:
rcondt 1; first auto.

to reduce the previous goal to:

In [ ]:
pre = d = A ∧ x = 1
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
while (d 6= Env) {
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
}
post = x = 5

(Here the argument 1 to rcondt refers to working on the first statement of the (one statement-long)
program, and auto is being applied to the first subgoal generated by running rcondt 1—the one that
pertains to the boolean expression. The remaining goal is the one we’re left to prove.)
Next, we apply


In [ ]:
rcondt 1; first auto

resulting in the goal

In [ ]:
pre = d = A ∧ x = 1
(d, x) <@ EntA.f(x);
while (d 6= Env) {
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
}
post = x = 5

We can then inline the first call of EntA.f

In [ ]:
inline (1) EntA.f.

yielding

In [ ]:
pre = d = A ∧ x = 1
x0 ← x;
x0 ← x0 + 1;
(d, x) ← (if 5 ≤ x0 then Env else B, x0);
while (d 6= Env) {
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
}
post = x = 5

Next, we can use the strongest postcondition tactic


In [ ]:
sp.

to push the initial assignments (the first three statements) into the precondition:

In [ ]:
pre =
exists (d0 : dest) (x1 : int),
x0 = x1 + 1 ∧
x = x0 ∧ d = if 5 ≤ x0 then Env else B ∧ d0 = A ∧ x1 = 1
while (d 6= Env) {
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
}
post = x = 5

Because the precondition now implies d = B, we can run


In [ ]:
rcondt 1; first auto.

getting us to

In [ ]:
pre =
exists (d0 : dest) (x1 : int),
x0 = x1 + 1 ∧
x = x0 ∧ d = if 5 ≤ x0 then Env else B ∧ d0 = A ∧ x1 = 1
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
while (d 6= Env) {
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
}
post = x = 5

We can then apply

In [ ]:
rcondf 1; first auto.

(note the “f” for a false boolean expression), yielding

In [ ]:
pre =
exists (d0 : dest) (x1 : int),
x0 = x1 + 1 ∧
x = x0 ∧ d = if 5 ≤ x0 then Env else B ∧ d0 = A ∧ x1 = 1
(d, x) <@ EntB.f(x);
while (d 6= Env) {
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
}
post = x = 5

Running

In [ ]:
inline (1) EntB.f.
sp.

will then take us to

In [ ]:
pre =
exists (d0 : dest) (x2 : int),
x1 = x2 ∗ 2 ∧
x = x1 ∧
d = if 5 ≤ x1 then Env else A ∧
exists (d1 : dest) (x3 : int),
x0 = x3 + 1 ∧
x2 = x0 ∧ d0 = if 5 ≤ x0 then Env else B ∧ d1 = A ∧ x3 = 1
while (d 6= Env) {
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
}
post = x = 5

Because the precondition now implies d = A, we can run

In [ ]:
rcondt 1; first auto.
rcondt 1; first auto.
inline (1) EntA.f.
sp

getting us to

In [ ]:
pre =
exists (d0 : dest) (x3 : int),
x2 = x3 + 1 ∧
x = x2 ∧
d = if 5 ≤ x2 then Env else B ∧
exists (d1 : dest) (x4 : int),
x1 = x4 ∗ 2 ∧
x3 = x1 ∧
d0 = if 5 ≤ x1 then Env else A ∧
exists (d2 : dest) (x5 : int),
x0 = x5 + 1 ∧
x4 = x0 ∧ d1 = if 5 ≤ x0 then Env else B ∧ d2 = A ∧ x5 = 1
while (d 6= Env) {
if (d = A) {
(d, x) <@ EntA.f(x);
}
else {
(d, x) <@ EntB.f(x);
}
}
post = x = 5

Because the precondition now implies d = Env, so that the loop’s boolean expression is now false,
we can run

In [ ]:
rcondf 1; first auto.

taking us to

In [ ]:
pre =
exists (d0 : dest) (x3 : int),
x2 = x3 + 1 ∧
x = x2 ∧
d = if 5 ≤ x2 then Env else B ∧
exists (d1 : dest) (x4 : int),
x1 = x4 ∗ 2 ∧
x3 = x1 ∧
d0 = if 5 ≤ x1 then Env else A ∧
exists (d2 : dest) (x5 : int),
x0 = x5 + 1 ∧
x4 = x0 ∧ d1 = if 5 ≤ x0 then Env else B ∧ d2 = A ∧ x5 = 1
post = x = 5

Finally, running

In [ ]:
auto

will solve this goal, completing the proof.
As the above symbolic evaluation proceeded, the preconditions became more and more layered.
It’s worth pointing out that EasyCrypt’s simplify tactic isn’t capable of making them simpler. But
in order to support symbolic evaluation in EasyCrypt, it will be helpful to implement a tactic for
more aggressively simplifying preconditions.
It’s also important to note that, when attempting to prove the truth or falsity of the boolean
expressions of conditionals and while loops, it’s often useful to employ SMT solvers. And when
doing this, one can supply a list of previously proved lemmas that the solvers may employ.
As argued in Section 6, we believe it will be possible to automate symbolic evaluation in EasyCrypt, making it trivial to prove lemmas like the one of this section, and resulting in succinct
proofs of such lemmas.